In [24]:
# ============================================================
# DAY 2 — SCREENER.IN SCRAPER + METRICS EXTRACTION
# AI Stock Screener — Indian Markets
# FINAL WORKING VERSION
# ============================================================
 
# ── CELL 1: IMPORTS ─────────────────────────────────────────
import pandas as pd
import numpy as np
import requests
import pickle
import time
import warnings
from bs4 import BeautifulSoup
warnings.filterwarnings('ignore')
 
print("Libraries loaded successfully!")

Libraries loaded successfully!


In [25]:
# ── CELL 2: SCRAPER FUNCTIONS ───────────────────────────────
 
def extract_table(table):
    """Extract any table from Screener.in cleanly"""
    try:
        thead   = table.find('thead')
        columns = []
        if thead:
            for th in thead.find_all('th'):
                text = th.get_text(strip=True)
                columns.append(text if text else 'Metric')
 
        tbody = table.find('tbody')
        if not tbody:
            return None
 
        rows = tbody.find_all('tr')
        data = {}
 
        for row in rows:
            cells       = row.find_all('td')
            if not cells:
                continue
            metric_name = cells[0].get_text(strip=True)
            metric_name = metric_name.replace('+', '').strip()
 
            skip_keywords = ['Raw PDF', 'PDF', 'Source']
            if any(kw in metric_name for kw in skip_keywords):
                continue
 
            values = []
            for cell in cells[1:]:
                val = cell.get_text(strip=True)
                val = val.replace(',', '').replace('%', '').strip()
                try:
                    values.append(float(val))
                except:
                    values.append(val if val else None)
 
            data[metric_name] = values
 
        if not data:
            return None
 
        col_names = columns[1:] if len(columns) > 1 else list(range(len(values)))
        df        = pd.DataFrame(data, index=col_names).T
        return df
 
    except Exception as e:
        print(f"Table extraction error: {e}")
        return None
 
 
def scrape_screener_complete(symbol):
    """
    Scrape all sections from Screener.in for a stock
    Tries consolidated first, falls back to standalone
    Returns dict with all sections as dataframes
    """
    try:
        # Try consolidated first
        url     = f"https://www.screener.in/company/{symbol}/consolidated/"
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
 
        response = requests.get(url, headers=headers, timeout=15)
 
        # Fallback to standalone
        if response.status_code != 200:
            url      = f"https://www.screener.in/company/{symbol}/"
            response = requests.get(url, headers=headers, timeout=15)
 
        if response.status_code != 200:
            return None
 
        soup     = BeautifulSoup(response.text, 'html.parser')
        sections = soup.find_all('section')
 
        target_sections = [
            'Quarterly Results',
            'Profit & Loss',
            'Balance Sheet',
            'Cash Flows',
            'Ratios',
            'Shareholding Pattern',
        ]
 
        result = {}
 
        for section in sections:
            heading = section.find('h2')
            if not heading:
                continue
            section_name = heading.text.strip()
            if section_name not in target_sections:
                continue
            table = section.find('table')
            if not table:
                continue
            df = extract_table(table)
            if df is not None:
                result[section_name] = df
 
        # Extract sector info from Peer Comparison section
        for section in sections:
            heading = section.find('h2')
            if heading and 'Peer' in heading.text:
                links       = section.find_all('a')
                sector_info = {}
                titles      = ['Broad Sector', 'Sector', 'Broad Industry', 'Industry']
                for link in links:
                    title = link.get('title', '')
                    if title in titles:
                        sector_info[title] = link.text.strip()
                result['Sector Info'] = sector_info
                break
 
        return result if result else None
 
    except Exception as e:
        print(f"Error scraping {symbol}: {e}")
        return None
 
 
print("Scraper functions loaded!")

Scraper functions loaded!


In [26]:
# ── CELL 3: FETCH ALL STOCKS ────────────────────────────────
 
stocks_to_fetch = [
    # Large Cap IT
    'TCS', 'INFY', 'WIPRO',
    # Mid Cap IT
    'PERSISTENT', 'MPHASIS', 'LTTS', 'COFORGE', 'HAPPSTMNDS',
    # Finance
    'CHOLAFIN', 'MUTHOOTFIN', 'MANAPPURAM', 'IIFL', 'CREDITACC',
    'HDFCBANK',
    # Manufacturing
    'GARFIBRES', 'SUPRAJIT', 'HIMATSEIDE', 'IGPL',
    # Defence
    'HAL', 'BEL', 'BHEL', 'MIDHANI', 'PARAS',
    # Chemicals
    'VINATIORGA', 'CLEAN', 'FINEORG', 'GALAXYSURF',
    # Finance/Housing
    'AAVAS',
    # Pharma
    'SUNPHARMA', 'DIVISLAB', 'APLLTD', 'GRANULES', 'IPCALAB',
    # Consumer
    'TITAN', 'HAVELLS', 'VGUARD', 'SYMPHONY', 'WONDERLA',
    'RELIANCE',
    # Agrochem
    'PIIND', 'RALLIS', 'DHANUKA',
    # Others
    'DYNPRO', 'GULFPETRO', 'SUVIDHAA', 'ABSLAMC',
]
 
all_stock_data = {}
failed_stocks  = []
 
print(f"Fetching data for {len(stocks_to_fetch)} stocks...")
print("This takes ~15-20 minutes. Please wait...\n")
 
for i, symbol in enumerate(stocks_to_fetch):
    try:
        print(f"[{i+1}/{len(stocks_to_fetch)}] {symbol}...", end=" ")
        data = scrape_screener_complete(symbol)
        if data:
            all_stock_data[symbol] = data
            sections = [k for k in data.keys() if k != 'Sector Info']
            print(f"✓ ({len(sections)} sections)")
        else:
            failed_stocks.append(symbol)
            print(f"✗ Failed")
        time.sleep(2)
    except Exception as e:
        failed_stocks.append(symbol)
        print(f"✗ Error: {e}")
        time.sleep(2)
 
print(f"\nFetched : {len(all_stock_data)} stocks")
print(f"Failed  : {len(failed_stocks)} — {failed_stocks}")
 
# Save raw data
with open('../data/raw_stock_data.pkl', 'wb') as f:
    pickle.dump(all_stock_data, f)
print("Saved raw_stock_data.pkl ✅")

Fetching data for 46 stocks...
This takes ~15-20 minutes. Please wait...

[1/46] TCS... ✓ (6 sections)
[2/46] INFY... ✓ (6 sections)
[3/46] WIPRO... ✓ (6 sections)
[4/46] PERSISTENT... ✓ (6 sections)
[5/46] MPHASIS... ✓ (6 sections)
[6/46] LTTS... ✓ (6 sections)
[7/46] COFORGE... ✓ (6 sections)
[8/46] HAPPSTMNDS... ✓ (6 sections)
[9/46] CHOLAFIN... ✓ (6 sections)
[10/46] MUTHOOTFIN... ✓ (6 sections)
[11/46] MANAPPURAM... ✓ (6 sections)
[12/46] IIFL... ✓ (6 sections)
[13/46] CREDITACC... ✓ (6 sections)
[14/46] HDFCBANK... ✓ (6 sections)
[15/46] GARFIBRES... ✓ (6 sections)
[16/46] SUPRAJIT... ✓ (6 sections)
[17/46] HIMATSEIDE... ✓ (6 sections)
[18/46] IGPL... ✓ (6 sections)
[19/46] HAL... ✓ (6 sections)
[20/46] BEL... ✓ (6 sections)
[21/46] BHEL... ✓ (6 sections)
[22/46] MIDHANI... ✓ (6 sections)
[23/46] PARAS... ✓ (6 sections)
[24/46] VINATIORGA... ✓ (6 sections)
[25/46] CLEAN... ✓ (6 sections)
[26/46] FINEORG... ✓ (6 sections)
[27/46] GALAXYSURF... ✓ (6 sections)
[28/46] AAVAS... ✓ (6 

In [27]:
# ── CELL 4: LOAD RAW DATA (if already fetched) ──────────────
with open('../data/raw_stock_data.pkl', 'rb') as f:
    all_stock_data = pickle.load(f)
print(f"Loaded: {len(all_stock_data)} stocks")

Loaded: 46 stocks


In [28]:
# ── CELL 5: EXTRACT ALL METRICS ─────────────────────────────
 
def safe_cagr(end_val, start_val, years):
    """Calculate CAGR safely"""
    try:
        if (end_val and start_val and
            start_val > 0 and end_val > 0 and years > 0):
            return round(((end_val / start_val) ** (1/years) - 1) * 100, 2)
    except:
        pass
    return None
 
 
def extract_all_metrics(symbol, data):
    """
    Extract all 50 key fundamental metrics from raw Screener.in data
    Handles both regular companies and financial companies differently
    """
    metrics = {'Symbol': symbol}
 
    try:
        # ── SECTOR INFO ──────────────────────────────────────
        sector_info             = data.get('Sector Info', {})
        metrics['Sector']       = sector_info.get('Sector', None)
        metrics['Industry']     = sector_info.get('Industry', None)
        metrics['Broad Sector'] = sector_info.get('Broad Sector', None)
 
        # Detect financial company
        financial_keywords = ['Financial', 'Banking', 'Insurance', 'NBFC']
        is_financial = any(
            kw in str(metrics.get('Sector', ''))
            for kw in financial_keywords
        )
 
        # ── PROFIT & LOSS ─────────────────────────────────────
        pl = data.get('Profit & Loss')
        if pl is not None:
            pl_clean = pl.drop(columns=['TTM'], errors='ignore')
 
            # Revenue
            rev_row = None
            for row_name in ['Sales', 'Revenue', 'Net Interest Income',
                             'Total Income', 'Financing Margin']:
                if row_name in pl_clean.index:
                    rev_row = row_name
                    break
 
            if rev_row:
                sales = pl_clean.loc[rev_row].dropna()
                sales_numeric = pd.to_numeric(sales, errors='coerce').dropna()
 
                if len(sales_numeric) > 0:
                    metrics['TTM Revenue']      = sales_numeric.iloc[-1]
                    metrics['Sales TTM']        = sales_numeric.iloc[-1]
 
                    # CAGRs
                    if len(sales_numeric) >= 6:
                        metrics['Revenue CAGR 5Y'] = safe_cagr(
                            sales_numeric.iloc[-1], sales_numeric.iloc[-6], 5
                        )
                    if len(sales_numeric) >= 11:
                        metrics['Revenue CAGR 10Y'] = safe_cagr(
                            sales_numeric.iloc[-1], sales_numeric.iloc[-11], 10
                        )
                    # Max CAGR
                    if len(sales_numeric) >= 2:
                        years_available = len(sales_numeric) - 1
                        metrics['Revenue CAGR Max']  = safe_cagr(
                            sales_numeric.iloc[-1], sales_numeric.iloc[0],
                            years_available
                        )
                        metrics['Revenue CAGR Years'] = years_available
 
                    # YoY quarterly growth
                    metrics['Revenue YoY Q'] = None
 
            # Net Profit
            profit_row = None
            for row_name in ['Net Profit', 'Profit after tax', 'PAT',
                             'Net profit']:
                if row_name in pl_clean.index:
                    profit_row = row_name
                    break
 
            if profit_row:
                profit = pl_clean.loc[profit_row].dropna()
                profit_numeric = pd.to_numeric(profit, errors='coerce').dropna()
 
                if len(profit_numeric) > 0:
                    metrics['TTM Profit']       = profit_numeric.iloc[-1]
                    metrics['Profit TTM']       = profit_numeric.iloc[-1]
 
                    if len(profit_numeric) >= 6:
                        metrics['Profit CAGR 5Y'] = safe_cagr(
                            profit_numeric.iloc[-1], profit_numeric.iloc[-6], 5
                        )
                    if len(profit_numeric) >= 11:
                        metrics['Profit CAGR 10Y'] = safe_cagr(
                            profit_numeric.iloc[-1], profit_numeric.iloc[-11], 10
                        )
                    if len(profit_numeric) >= 2:
                        years_available = len(profit_numeric) - 1
                        metrics['Profit CAGR Max']  = safe_cagr(
                            profit_numeric.iloc[-1], profit_numeric.iloc[0],
                            years_available
                        )
                        metrics['Profit CAGR Years'] = years_available
 
            # OPM
            for opm_name in ['OPM %', 'OPM', 'Operating Profit Margin']:
                if opm_name in pl_clean.index:
                    opm = pd.to_numeric(
                        pl_clean.loc[opm_name], errors='coerce'
                    ).dropna()
                    if len(opm) > 0:
                        metrics['Avg OPM 5Y']  = round(
                            opm.tail(5).mean(), 1
                        ) if len(opm) >= 5 else round(opm.mean(), 1)
                        metrics['Avg OPM 10Y'] = round(
                            opm.tail(10).mean(), 1
                        ) if len(opm) >= 10 else round(opm.mean(), 1)
                    break
 
        # ── QUARTERLY RESULTS ────────────────────────────────
        qr = data.get('Quarterly Results')
        if qr is not None:
            # Revenue quarterly trend
            rev_q_row = None
            for row_name in ['Sales', 'Revenue', 'Net Interest Income',
                             'Total Income']:
                if row_name in qr.index:
                    rev_q_row = row_name
                    break
 
            if rev_q_row:
                rev_q = pd.to_numeric(
                    qr.loc[rev_q_row], errors='coerce'
                ).dropna()
                if len(rev_q) >= 5:
                    metrics['Revenue YoY Q'] = round(
                        (rev_q.iloc[-1] / rev_q.iloc[-5] - 1) * 100, 2
                    ) if rev_q.iloc[-5] > 0 else None
                    # Count consecutive YoY growth
                    consecutive = 0
                    for i in range(1, min(len(rev_q), 5)):
                        if (len(rev_q) > i+3 and
                            rev_q.iloc[-(i)] > rev_q.iloc[-(i+4)]):
                            consecutive += 1
                        else:
                            break
                    metrics['Revenue Consecutive YoY'] = consecutive
                    metrics['Revenue Accelerating'] = (
                        len(rev_q) >= 6 and
                        (rev_q.iloc[-1]/rev_q.iloc[-5] >
                         rev_q.iloc[-2]/rev_q.iloc[-6])
                    ) if len(rev_q) >= 6 else False
 
            # Profit quarterly
            profit_q_row = None
            for row_name in ['Net Profit', 'Profit after tax', 'PAT']:
                if row_name in qr.index:
                    profit_q_row = row_name
                    break
 
            if profit_q_row:
                profit_q = pd.to_numeric(
                    qr.loc[profit_q_row], errors='coerce'
                ).dropna()
                if len(profit_q) > 0:
                    metrics['Profit YoY Q']     = round(
                        (profit_q.iloc[-1] / profit_q.iloc[-5] - 1) * 100, 2
                    ) if len(profit_q) >= 5 and profit_q.iloc[-5] > 0 else None
                    metrics['Profit Positive Q'] = int((profit_q > 0).sum())
                    metrics['Total Quarters']    = len(profit_q)
 
            # OPM quarterly
            for opm_name in ['OPM %', 'OPM']:
                if opm_name in qr.index:
                    opm_q = pd.to_numeric(
                        qr.loc[opm_name], errors='coerce'
                    ).dropna()
                    if len(opm_q) > 0:
                        metrics['Latest OPM Q']   = opm_q.iloc[-1]
                        metrics['Avg OPM 4Q']     = round(
                            opm_q.tail(4).mean(), 2
                        ) if len(opm_q) >= 4 else round(opm_q.mean(), 2)
                        metrics['Margin Improving'] = (
                            opm_q.iloc[-1] > opm_q.iloc[-4]
                        ) if len(opm_q) >= 4 else None
                    break
 
            # EPS
            for eps_name in ['EPS in Rs', 'EPS', 'Basic EPS']:
                if eps_name in qr.index:
                    eps_q = pd.to_numeric(
                        qr.loc[eps_name], errors='coerce'
                    ).dropna()
                    if len(eps_q) > 0:
                        metrics['Latest EPS Q']   = eps_q.iloc[-1]
                        metrics['EPS YoY Growth'] = round(
                            (eps_q.iloc[-1] / eps_q.iloc[-5] - 1) * 100, 2
                        ) if len(eps_q) >= 5 and eps_q.iloc[-5] != 0 else None
                    break
 
        # ── BALANCE SHEET ────────────────────────────────────
        bs = data.get('Balance Sheet')
        if bs is not None:
            # Borrowings
            for debt_name in ['Borrowings', 'Total Liabilities',
                              'Borrowing']:
                if debt_name in bs.index:
                    debt = pd.to_numeric(
                        bs.loc[debt_name], errors='coerce'
                    ).dropna()
                    if len(debt) > 0:
                        metrics['Latest Debt']  = debt.iloc[-1]
                        metrics['Debt Reducing'] = (
                            debt.iloc[-1] < debt.iloc[-3]
                        ) if len(debt) >= 3 else None
                    break
 
            # Equity
            equity_total = None
            if 'Reserves' in bs.index and 'Equity Capital' in bs.index:
                reserves   = pd.to_numeric(
                    bs.loc['Reserves'], errors='coerce'
                ).dropna()
                eq_capital = pd.to_numeric(
                    bs.loc['Equity Capital'], errors='coerce'
                ).dropna()
                if len(reserves) > 0 and len(eq_capital) > 0:
                    equity_total             = (
                        reserves.iloc[-1] + eq_capital.iloc[-1]
                    )
                    metrics['Latest Equity'] = equity_total
 
            # Debt/Equity
            if metrics.get('Latest Debt') and equity_total and equity_total > 0:
                metrics['Debt to Equity'] = round(
                    metrics['Latest Debt'] / equity_total, 2
                )
 
            # ROE from balance sheet
            if metrics.get('TTM Profit') and equity_total and equity_total > 0:
                metrics['ROE'] = round(
                    metrics['TTM Profit'] / equity_total * 100, 2
                )
 
        # ── CASH FLOWS ───────────────────────────────────────
        cf = data.get('Cash Flows')
        if cf is not None:
            for cf_name in ['Cash from Operating Activity',
                            'Operating Activity', 'Cash from Operations']:
                if cf_name in cf.index:
                    op_cf = pd.to_numeric(
                        cf.loc[cf_name], errors='coerce'
                    ).dropna()
                    if len(op_cf) > 0:
                        metrics['Latest Operating CF'] = op_cf.iloc[-1]
                        metrics['CF Positive Years']   = int((op_cf > 0).sum())
                        metrics['CF Total Years']      = len(op_cf)
                        metrics['CF Growing']          = (
                            op_cf.iloc[-1] > op_cf.iloc[-3]
                        ) if len(op_cf) >= 3 else None
 
                        # CF consistency (% of positive years)
                        if len(op_cf) > 0:
                            metrics['CF Consistency'] = round(
                                int((op_cf > 0).sum()) / len(op_cf) * 100, 1
                            )
                    break
 
        # ── RATIOS ───────────────────────────────────────────
        ratios = data.get('Ratios')
        if ratios is not None:
            # ROCE
            for roce_name in ['ROCE %', 'Return on Capital Employed',
                              'ROCE']:
                if roce_name in ratios.index:
                    roce = pd.to_numeric(
                        ratios.loc[roce_name], errors='coerce'
                    ).dropna()
                    if len(roce) > 0:
                        metrics['Latest ROCE']    = roce.iloc[-1]
                        metrics['ROCE Improving'] = (
                            roce.iloc[-1] > roce.iloc[-3]
                        ) if len(roce) >= 3 else None
                        metrics['Avg ROCE 5Y']    = round(
                            roce.tail(5).mean(), 1
                        ) if len(roce) >= 5 else round(roce.mean(), 1)
                    break
 
            # ROE from Ratios (more reliable than calculated)
            for roe_name in ['ROE %', 'Return on Equity', 'ROE']:
                if roe_name in ratios.index:
                    roe_r = pd.to_numeric(
                        ratios.loc[roe_name], errors='coerce'
                    ).dropna()
                    if len(roe_r) > 0:
                        metrics['ROE from Ratios'] = roe_r.iloc[-1]
                        metrics['ROE Avg 5Y']      = round(
                            roe_r.tail(5).mean(), 1
                        ) if len(roe_r) >= 5 else round(roe_r.mean(), 1)
                        metrics['ROE Improving']   = (
                            roe_r.iloc[-1] > roe_r.iloc[-3]
                        ) if len(roe_r) >= 3 else None
                    break
 
            # PE Ratio
            for pe_name in ['Price to Earning', 'PE Ratio', 'P/E']:
                if pe_name in ratios.index:
                    pe = pd.to_numeric(
                        ratios.loc[pe_name], errors='coerce'
                    ).dropna()
                    if len(pe) > 0:
                        metrics['PE Ratio'] = pe.iloc[-1]
                    break
 
        # ── FINAL ROE (unified) ───────────────────────────────
        # Use ROE from Ratios if available
        # For financial companies ROE from Ratios is more accurate
        if metrics.get('ROE from Ratios'):
            metrics['Final ROE'] = metrics['ROE from Ratios']
        elif metrics.get('ROE'):
            metrics['Final ROE'] = metrics['ROE']
        else:
            metrics['Final ROE'] = None
 
        # ── SHAREHOLDING ─────────────────────────────────────
        sh = data.get('Shareholding Pattern')
        if sh is not None:
            # Promoter holding
            for promo_name in ['Promoters', 'Promoter']:
                if promo_name in sh.index:
                    promoter = pd.to_numeric(
                        sh.loc[promo_name], errors='coerce'
                    ).dropna()
                    if len(promoter) > 0:
                        metrics['Promoter Holding']  = promoter.iloc[-1]
                        # Change over last 4 quarters
                        metrics['Promoter Change 4Q'] = round(
                            promoter.iloc[-1] - promoter.iloc[-5], 2
                        ) if len(promoter) >= 5 else 0
                        # Change over last 8 quarters
                        metrics['Promoter Change 8Q'] = round(
                            promoter.iloc[-1] - promoter.iloc[-9], 2
                        ) if len(promoter) >= 9 else (
                            round(promoter.iloc[-1] - promoter.iloc[0], 2)
                            if len(promoter) > 1 else 0
                        )
                    break
 
            # FII Holding
            for fii_name in ['FIIs', 'FII', 'Foreign Institutions']:
                if fii_name in sh.index:
                    fii = pd.to_numeric(
                        sh.loc[fii_name], errors='coerce'
                    ).dropna()
                    if len(fii) > 0:
                        metrics['FII Holding']    = fii.iloc[-1]
                        metrics['FII Change 4Q']  = round(
                            fii.iloc[-1] - fii.iloc[-5], 2
                        ) if len(fii) >= 5 else 0
                    break
 
            # DII Holding
            for dii_name in ['DIIs', 'DII', 'Domestic Institutions']:
                if dii_name in sh.index:
                    dii = pd.to_numeric(
                        sh.loc[dii_name], errors='coerce'
                    ).dropna()
                    if len(dii) > 0:
                        metrics['DII Holding']    = dii.iloc[-1]
                        metrics['DII Change 4Q']  = round(
                            dii.iloc[-1] - dii.iloc[-5], 2
                        ) if len(dii) >= 5 else 0
                    break
 
    except Exception as e:
        print(f"Error extracting metrics for {symbol}: {e}")
 
    return metrics
 
 
# Test on TCS
test = extract_all_metrics('TCS', all_stock_data['TCS'])
print("=== TCS METRICS ===")
for key, value in test.items():
    print(f"  {key:30} : {value}")

=== TCS METRICS ===
  Symbol                         : TCS
  Sector                         : Information Technology
  Industry                       : Computers - Software & Consulting
  Broad Sector                   : Information Technology
  TTM Revenue                    : 255324.0
  Sales TTM                      : 255324.0
  Revenue CAGR 5Y                : 10.22
  Revenue CAGR 10Y               : 10.43
  Revenue CAGR Max               : 10.9
  Revenue CAGR Years             : 11
  Revenue YoY Q                  : 4.87
  TTM Profit                     : 48797.0
  Profit TTM                     : 48797.0
  Profit CAGR 5Y                 : 8.5
  Profit CAGR 10Y                : 9.3
  Profit CAGR Max                : 8.78
  Profit CAGR Years              : 11
  Avg OPM 5Y                     : 27.0
  Avg OPM 10Y                    : 27.0
  Revenue Consecutive YoY        : 4
  Revenue Accelerating           : True
  Profit YoY Q                   : -13.85
  Profit Positive Q        

In [29]:
# ── CELL 6: EXTRACT FOR ALL STOCKS ──────────────────────────
all_metrics = []
 
for symbol, data in all_stock_data.items():
    metrics = extract_all_metrics(symbol, data)
    all_metrics.append(metrics)
    print(f"✓ {symbol}")
 
metrics_df = pd.DataFrame(all_metrics)
metrics_df.to_csv('../data/fundamental_metrics.csv', index=False)
 
print(f"\nTotal stocks    : {len(metrics_df)}")
print(f"Metrics per stock: {len(metrics_df.columns)}")
print(f"\nColumns available:")
for col in metrics_df.columns.tolist():
    print(f"  {col}")

✓ TCS
✓ INFY
✓ WIPRO
✓ PERSISTENT
✓ MPHASIS
✓ LTTS
✓ COFORGE
✓ HAPPSTMNDS
✓ CHOLAFIN
✓ MUTHOOTFIN
✓ MANAPPURAM
✓ IIFL
✓ CREDITACC
✓ HDFCBANK
✓ GARFIBRES
✓ SUPRAJIT
✓ HIMATSEIDE
✓ IGPL
✓ HAL
✓ BEL
✓ BHEL
✓ MIDHANI
✓ PARAS
✓ VINATIORGA
✓ CLEAN
✓ FINEORG
✓ GALAXYSURF
✓ AAVAS
✓ SUNPHARMA
✓ DIVISLAB
✓ APLLTD
✓ GRANULES
✓ IPCALAB
✓ TITAN
✓ HAVELLS
✓ VGUARD
✓ SYMPHONY
✓ WONDERLA
✓ RELIANCE
✓ PIIND
✓ RALLIS
✓ DHANUKA
✓ DYNPRO
✓ GULFPETRO
✓ SUVIDHAA
✓ ABSLAMC

Total stocks    : 46
Metrics per stock: 53

Columns available:
  Symbol
  Sector
  Industry
  Broad Sector
  TTM Revenue
  Sales TTM
  Revenue CAGR 5Y
  Revenue CAGR 10Y
  Revenue CAGR Max
  Revenue CAGR Years
  Revenue YoY Q
  TTM Profit
  Profit TTM
  Profit CAGR 5Y
  Profit CAGR 10Y
  Profit CAGR Max
  Profit CAGR Years
  Avg OPM 5Y
  Avg OPM 10Y
  Revenue Consecutive YoY
  Revenue Accelerating
  Profit YoY Q
  Profit Positive Q
  Total Quarters
  Latest OPM Q
  Avg OPM 4Q
  Margin Improving
  Latest EPS Q
  EPS YoY Growth
  Latest Deb